### Step 1: Import API keys and libraries

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API_KEY is missing.")
else:
    print(OPENAI_API_KEY[:8])
    
client = OpenAI()

sk-proj-


### Step 2: Set up Pushover

In [2]:
load_dotenv()

True

In [3]:
pushover_user = os.getenv("PUSHOEVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
import requests 
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
#send_notification("Hello from AI Challenge!")

### Step 3: Describe Pushover as an LLM tool

In [6]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a notification to the user via Pushover.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send as a notification."
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for the LLM

In [7]:
tools = [{"type": "function", "function":send_notification_function}]

### Step 5: Calling the tool from an LLM

In [16]:
def handle_tool_call(tool_call):
    tool_call = tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    send_notification(args["message"])
    
    tool_call_result = {
        "role": "tool",
        "content": f"Notification sent with message: {args['message']}",
        "tool_call_id": tool_call.id
    }
    
    return tool_call_result

In [17]:
client = OpenAI()

messages=[{"role": "user", "content": "Please send me a notification telling me what amazing progress\
        I'm making on the AI Engineering training by SuperDataScience."}]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

#Check if model wants to call a tool
message = response.choices[0].message

if message.tool_calls:
    tool_result = handle_tool_call(message.tool_calls)
    messages.append(message)
    messages.append(tool_result)
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
        #tools=tools
    )
    
    message = response.choices[0].message    
    
print(message.content)


NameError: name 'tool_calls' is not defined